In [1]:
from credentials.params.session_parameters import connection_parameters
from snowflake.snowpark import Session

session = Session.builder.configs(connection_parameters).create()

In [5]:
session.sql("USE WAREHOUSE COMPUTE_WH").collect()
session.sql("USE DATABASE SUPPLY_CHAIN_DB").collect()
trans_df = session.table("TRANSFORMED.STG_ORDERS")
trans_df.show()

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
from snowflake.snowpark import functions as F

# Basic row / schema overview
print("Row count:", trans_df.count())
print("\nSchema:")
print(trans_df.schema)
print("\nSample rows:")
trans_df.limit(10).show()

Row count: 180519

Schema:
StructType([StructField('TYPE', StringType(16777216), nullable=True), StructField('DAYS_FOR_SHIPPING_REAL', LongType(), nullable=True), StructField('DAYS_FOR_SHIPMENT_SCHEDULED', LongType(), nullable=True), StructField('BENEFIT_PER_ORDER', DecimalType(13, 9), nullable=True), StructField('SALES_PER_CUSTOMER', DecimalType(13, 9), nullable=True), StructField('DELIVERY_STATUS', StringType(16777216), nullable=True), StructField('LATE_DELIVERY_RISK', LongType(), nullable=True), StructField('CATEGORY_ID', StringType(16777216), nullable=True), StructField('CATEGORY_NAME', StringType(16777216), nullable=True), StructField('CUSTOMER_CITY', StringType(16777216), nullable=True), StructField('CUSTOMER_COUNTRY', StringType(16777216), nullable=True), StructField('CUSTOMER_EMAIL', StringType(16777216), nullable=True), StructField('CUSTOMER_FNAME', StringType(16777216), nullable=True), StructField('CUSTOMER_ID', StringType(16777216), nullable=True), StructField('CUSTOMER_LNAM

In [7]:
# Summary stats (built-in describe covers numeric columns)
print("\nSummary statistics:")
try:
    trans_df.describe().show()
except Exception as e:
    print("describe() failed:", e)


Summary statistics:
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [8]:
# Null counts per column
cols = [f.name for f in trans_df.schema.fields]
null_exprs = [F.sum(F.when(F.col(c).is_null(), 1).otherwise(0)).alias(c) for c in cols]
print("\nNull counts per column:")
try:
    trans_df.select(null_exprs).show()
except Exception as e:
    print("Null-count aggregation failed:", e)


Null counts per column:
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [9]:
# Distinct counts per column
distinct_exprs = [F.count_distinct(F.col(c)).alias(c) for c in cols]
print("\nDistinct counts per column:")
try:
    trans_df.select(distinct_exprs).show()
except Exception as e:
    print("Distinct-count aggregation failed:", e)


Distinct counts per column:
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [10]:
# Top values for up to first 5 columns (useful for categorical inspection)
print("\nTop values (up to 5 columns):")
for c in cols[:5]:
    print(f"\nTop values for column: {c}")
    try:
        trans_df.group_by(c).count().sort(F.col("COUNT").desc()).limit(10).show()
    except Exception as e:
        print(f"Failed to compute top values for {c}: {e}")


Top values (up to 5 columns):

Top values for column: TYPE
----------------------
|"TYPE"    |"COUNT"  |
----------------------
|DEBIT     |69295    |
|TRANSFER  |49883    |
|PAYMENT   |41725    |
|CASH      |19616    |
----------------------


Top values for column: DAYS_FOR_SHIPPING_REAL
--------------------------------------
|"DAYS_FOR_SHIPPING_REAL"  |"COUNT"  |
--------------------------------------
|2                         |56618    |
|3                         |28765    |
|6                         |28723    |
|4                         |28513    |
|5                         |28163    |
|0                         |5080     |
|1                         |4657     |
--------------------------------------


Top values for column: DAYS_FOR_SHIPMENT_SCHEDULED
-------------------------------------------
|"DAYS_FOR_SHIPMENT_SCHEDULED"  |"COUNT"  |
-------------------------------------------
|4                              |107752   |
|2                              |35216    |
|1    